In [ ]:
import pandas as pd


# ============================================================
# 1. LOAD FALSE POSITIVE AND FALSE NEGATIVE CSV FILES
# ============================================================

fp = pd.read_csv("false_positives.csv")
fn = pd.read_csv("false_negatives.csv")


# ============================================================
# 2. FUNCTION TO ANALYZE REASONS
# ============================================================

def analyze_reasons(df, reasons, title, output_file):

    total = len(df)

    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    print(f"Total records: {total}\n")

    summary_results = []
    detailed_results = []

    # --------------------------------------------------------
    # Check every reason
    # --------------------------------------------------------

    for reason, condition in reasons.items():

        # Boolean mask identifying rows matching this reason
        mask = condition(df)

        # Number of occurrences
        count = mask.sum()

        # Percentage
        percentage = (count / total) * 100 if total > 0 else 0

        # Store summary information
        summary_results.append({
            "Reason": reason,
            "Occurrences": count,
            "Percentage": round(percentage, 2)
        })

        # ----------------------------------------------------
        # Store every transaction that matches this reason
        # ----------------------------------------------------

        matching_rows = df[mask].copy()

        # Add the reason responsible for the occurrence
        matching_rows["Reason"] = reason

        detailed_results.append(matching_rows)

    # ========================================================
    # 3. CREATE SUMMARY TABLE
    # ========================================================

    summary_df = pd.DataFrame(summary_results)

    print("SUMMARY")
    print("-" * 80)

    print(summary_df.to_string(index=False))


    # ========================================================
    # 4. CREATE DETAILED TABLE
    # ========================================================

    if detailed_results:

        detailed_df = pd.concat(
            detailed_results,
            ignore_index=True
        )

    else:

        detailed_df = pd.DataFrame()


    # ========================================================
    # 5. SAVE DETAILED RESULTS
    # ========================================================

    detailed_df.to_csv(
        output_file,
        index=False
    )


    # ========================================================
    # 6. SAVE SUMMARY RESULTS
    # ========================================================

    summary_file = output_file.replace(
        ".csv",
        "_summary.csv"
    )

    summary_df.to_csv(
        summary_file,
        index=False
    )


    # ========================================================
    # 7. DISPLAY INFORMATION
    # ========================================================

    print("\n" + "=" * 80)
    print("DETAILED OCCURRENCES")
    print("=" * 80)

    print(
        f"Detailed results saved to: {output_file}"
    )

    print(
        f"Summary results saved to: {summary_file}"
    )

    return summary_df, detailed_df


# ============================================================
# 8. DEFINE FALSE POSITIVE REASONS
# ============================================================
#
# CHANGE THESE REASONS ACCORDING TO YOUR DATASET.
# ============================================================

fp_reasons = {

    "International transaction":
        lambda df:
        df["is_international"] == 1,

    "Previous fraud history":
        lambda df:
        df["past_fraud_count"] >= 1,

    "Amount higher than average spending":
        lambda df:
        df["amount"] > df["avg_spend"],

    "Travel category":
        lambda df:
        df["category"].str.lower() == "travel",

    "Electronics category":
        lambda df:
        df["category"].str.lower() == "electronics",

    "Food category":
        lambda df:
        df["category"].str.lower() == "food",

    "Shopping category":
        lambda df:
        df["category"].str.lower() == "shopping",

    "Grocery category":
        lambda df:
        df["category"].str.lower() == "grocery",

}


# ============================================================
# 9. DEFINE FALSE NEGATIVE REASONS
# ============================================================

fn_reasons = {

    "Domestic transaction":
        lambda df:
        df["is_international"] == 0,

    "Moderate/normal transaction amount":
        lambda df:
        df["amount"] <= df["avg_spend"],

    "Walmart merchant":
        lambda df:
        df["merchant"].str.lower() == "walmart",

    "Previous fraud count = 1":
        lambda df:
        df["past_fraud_count"] == 1,

    "Shopping category":
        lambda df:
        df["category"].str.lower() == "shopping",

    "Grocery category":
        lambda df:
        df["category"].str.lower() == "grocery",

}


# ============================================================
# 10. ANALYZE FALSE POSITIVES
# ============================================================

fp_summary, fp_details = analyze_reasons(
    fp,
    fp_reasons,
    "FALSE POSITIVE ANALYSIS",
    "fp_detailed_reason_occurrences.csv"
)


# ============================================================
# 11. ANALYZE FALSE NEGATIVES
# ============================================================

fn_summary, fn_details = analyze_reasons(
    fn,
    fn_reasons,
    "FALSE NEGATIVE ANALYSIS",
    "fn_detailed_reason_occurrences.csv"
)


# ============================================================
# 12. FINISHED
# ============================================================

print("\n" + "=" * 80)
print("ANALYSIS COMPLETE")
print("=" * 80)

print("\nFiles created:")

print("1. fp_detailed_reason_occurrences.csv")
print("2. fp_detailed_reason_occurrences_summary.csv")

print("3. fn_detailed_reason_occurrences.csv")
print("4. fn_detailed_reason_occurrences_summary.csv")



FALSE POSITIVE ANALYSIS
Total records: 57

SUMMARY
--------------------------------------------------------------------------------
                             Reason  Occurrences  Percentage
          International transaction           35       61.40
             Previous fraud history           44       77.19
Amount higher than average spending           19       33.33
                    Travel category           19       33.33
               Electronics category           15       26.32
                      Food category           10       17.54
                  Shopping category            4        7.02
                   Grocery category            3        5.26

DETAILED OCCURRENCES
Detailed results saved to: fp_detailed_reason_occurrences.csv
Summary results saved to: fp_detailed_reason_occurrences_summary.csv

FALSE NEGATIVE ANALYSIS
Total records: 2

SUMMARY
--------------------------------------------------------------------------------
                            Reaso